# NHA Hackathon – Problem Statement 3  
## Skeletal Notebook: Document Forgery / Deepfake Detection

## Problem Understanding

According to the uploaded guidelines, PS3 requires participants to detect visible document tampering, classify each image/page using the predefined category IDs, and highlight manipulated regions using YAML annotations. The submission must contain:
1. a per-document or per-page **YAML annotation file**, and
2. a **JSON output** with `link`, `file_name`, and `Category_ID`.  

PDFs must be converted into per-page JPG-style entries such as `xyz_page_1.jpg` and produce corresponding YAML files such as `xyz_page_1.yaml`. Categories C8 and C10 are category-only cases with no YAML requirement, while other categories require bounding boxes in the exact YAML syntax expected by the organizers. fileciteturn1file0L1-L19 fileciteturn1file0L20-L30

## Categories in Scope

Use only the predefined category IDs:
- `C1` Copy-paste within the same document
- `C2` Overwriting on existing text
- `C3` Adding new content
- `C4` Removing / erasing text or image
- `C5` Merging content from different documents
- `C6` Watermark removal
- `C7` Irregular spacing
- `C8` Fully AI-generated document
- `C9` Partial AI-generated edits
- `C10` No editing / discrepancy found fileciteturn1file0L20-L30

In [2]:
# =========================
# 1. IMPORTS
# =========================
# =========================
# 0. ENVIRONMENT SETUP (Zero-GPU Stack)
# =========================
# Run this once to ensure the CPU instance has our exact required libraries.
# Using opencv-python-headless prevents UI dependencies from crashing the headless CPU instance.
!pip install -q opencv-python-headless pytesseract numpy scikit-learn scipy onnxruntime PyMuPDF PyYAML pandas pillow

# =========================
# 1. IMPORTS
# =========================

import os
import io
import json
import math
import base64
import asyncio
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# --- Our Custom Architecture Imports ---
import cv2                  # For Tier 2 (SIFT/ORB) and Tier 3 (ELA Contours)
import numpy as np          # For Tier 1 (IoU Math) and Array operations
import pytesseract          # For Tier 1 (C2 Overwriting, C7 Irregular Spacing)
import scipy.fftpack        # For Tier 4 (Fast Fourier Transform for AI artifacts)

try:
    import onnxruntime as ort # For Tier 4 (Quantized INT8 Inference)
except ImportError:
    ort = None

try:
    import yaml
except ImportError:
    yaml = None

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from PIL import Image, ImageChops, ImageEnhance # ImageChops is crucial for ELA
except ImportError:
    Image = None

try:
    import fitz  # PyMuPDF for secure, local PDF-to-Image conversion
except ImportError:
    fitz = None

# IMPORTANT: If testing on a local Windows machine, uncomment and set your Tesseract path.
# If running on the hackathon's cloud Linux instance, leave this commented out.
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

print("Phase 1: Environment Setup & Imports Successful.")

Phase 1: Environment Setup & Imports Successful.


## Download the Dataset
We have provided a dedicated widget to download the hackathon datasets directly from the platform into this notebook environment.

### 1. Import the Widget

from databank_download_widget import DatabankDownloadWidget

### 2. Download the Databank
Select the cell below and run it.
Enter the Databank ID for the hackathon package.
Enter your email and password for the platform.
Click the **Download** button.

The widget will download and unzip the data right into your current directory. You can monitor the progress in the status output area below the button.

### **Databank ID for PS3: 1ae9a4db-6b53-4a17-8274-fd3818c3f2be**

In [ ]:
from databank_download_widget import DatabankDownloadWidget

# Create an instance of the databank widget
databank_downloader = DatabankDownloadWidget()

# Display the widget
databank_downloader.display()

In [3]:
# =========================
# 2. PATHS & CONFIG
# =========================

INPUT_DIR = Path("./Claim_Documents")
OUTPUT_DIR = Path("./outputs")
ANNOTATIONS_DIR = OUTPUT_DIR / "annotations"
DEBUG_DIR = OUTPUT_DIR / "debug"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SUPPORTED_PDF_EXTS = {".pdf"}

CATEGORY_IDS = {
    "C1": "Copy-paste within the same document",
    "C2": "Overwriting on existing text",
    "C3": "Adding new content",
    "C4": "Removing / erasing text or image",
    "C5": "Merging content from different documents",
    "C6": "Watermark removal",
    "C7": "Irregular spacing",
    "C8": "Fully AI-generated document",
    "C9": "Partial AI-generated edits",
    "C10": "No editing / discrepancy found",
}

CATEGORY_ONLY_CLASSES = {"C8", "C10"}

In [4]:
# =========================
# 4. DATA STRUCTURES
# =========================

@dataclass
class DocumentPage:
    source_link: str
    original_path: str
    original_file_name: str
    page_file_name: str
    page_number: int
    is_pdf: bool
    image_path: Optional[str] = None
    image_width: Optional[int] = None
    image_height: Optional[int] = None


@dataclass
class DetectedRegion:
    x: int
    y: int
    w: int
    h: int
    category_id: str
    type: Optional[str] = None
    stretch_factor: Optional[float] = None
    header_source: Optional[str] = None
    body_source: Optional[str] = None


@dataclass
class PageAnalysisResult:
    source_link: str
    file_name: str
    original_file_name: str
    page_number: int
    predicted_categories: List[str] = field(default_factory=list)
    detected_regions: List[DetectedRegion] = field(default_factory=list)
    notes: Dict[str, Any] = field(default_factory=dict)
    debug: Dict[str, Any] = field(default_factory=dict)

In [6]:
# =========================
# DATA ONBOARDING HELPERS
# =========================

def safe_open_image_size(image_path: Path) -> Tuple[Optional[int], Optional[int]]:
    if Image is None:
        return None, None
    try:
        with Image.open(image_path) as img:
            return img.size
    except Exception:
        return None, None


def render_pdf_to_images(pdf_path: Path, render_dir: Path) -> List[DocumentPage]:
    pages: List[DocumentPage] = []

    if fitz is None:
        raise ImportError("PyMuPDF (fitz) is required to render PDF pages for PS3 onboarding.")

    doc = fitz.open(pdf_path)
    try:
        for idx in range(len(doc)):
            page = doc.load_page(idx)
            pix = page.get_pixmap()
            page_file_name = f"{pdf_path.stem}_page_{idx + 1}.jpg"
            image_path = render_dir / page_file_name
            pix.save(str(image_path))

            width, height = safe_open_image_size(image_path)
            pages.append(
                DocumentPage(
                    source_link=str(pdf_path),
                    original_path=str(pdf_path),
                    original_file_name=pdf_path.name,
                    page_file_name=page_file_name,
                    page_number=idx + 1,
                    is_pdf=True,
                    image_path=str(image_path),
                    image_width=width,
                    image_height=height,
                )
            )
    finally:
        doc.close()

    return pages


def build_document_pages(input_dir: Path, render_dir: Path) -> List[DocumentPage]:
    pages: List[DocumentPage] = []
    render_dir.mkdir(parents=True, exist_ok=True)

    for file_path in sorted(input_dir.iterdir()):
        if not file_path.is_file():
            continue

        suffix = file_path.suffix.lower()

        if suffix in SUPPORTED_IMAGE_EXTS:
            width, height = safe_open_image_size(file_path)
            pages.append(
                DocumentPage(
                    source_link=str(file_path),
                    original_path=str(file_path),
                    original_file_name=file_path.name,
                    page_file_name=file_path.name,
                    page_number=1,
                    is_pdf=False,
                    image_path=str(file_path),
                    image_width=width,
                    image_height=height,
                )
            )
        elif suffix in SUPPORTED_PDF_EXTS:
            pages.extend(render_pdf_to_images(file_path, render_dir))

    return pages

In [7]:
import cv2
import numpy as np
import os
from PIL import Image, ImageChops, ImageEnhance
from typing import List

def run_tier3_ela_contours(page: DocumentPage) -> List[DetectedRegion]:
    """
    Tier 3 Scanner: Targets C3 (Added), C4 (Erased), and C6 (Watermark Removed).
    Generates an Error Level Analysis map and uses OpenCV contours to find anomalies.
    """
    regions = []
    if not page.image_path:
        return regions
        
    try:
        # 1. GENERATE THE ELA MAP
        original = Image.open(page.image_path).convert('RGB')
        
        # Save a temporary compressed version (quality 90 triggers ELA differences)
        temp_filename = f"temp_ela_{page.page_file_name}.jpg"
        original.save(temp_filename, 'JPEG', quality=90)
        compressed = Image.open(temp_filename)
        
        # Calculate the absolute mathematical difference between original and compressed
        ela_diff = ImageChops.difference(original, compressed)
        
        # Enhance the difference significantly so anomalies glow brightly
        extrema = ela_diff.getextrema()
        max_diff = max([ex[1] for ex in extrema])
        if max_diff == 0:
            max_diff = 1
        scale = 255.0 / max_diff
        ela_map = ImageEnhance.Brightness(ela_diff).enhance(scale)
        
        # Cleanup temp file
        os.remove(temp_filename)
        
        # 2. OPENCV CONTOUR DETECTION
        # Convert ELA map to OpenCV format (Grayscale)
        ela_cv = np.array(ela_map)
        ela_gray = cv2.cvtColor(ela_cv, cv2.COLOR_RGB2GRAY)
        
        # Apply strict binary threshold: Only the top 5% brightest pixels survive
        _, thresh = cv2.threshold(ela_gray, 240, 255, cv2.THRESH_BINARY)
        
        # Morphological operations to close gaps and smooth out noise
        kernel = np.ones((5,5), np.uint8)
        thresh = cv2.dilate(thresh, kernel, iterations=2)
        
        # Find the physical boundaries of the glowing spots
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # 3. THE CONTEXT ENGINE (Classifying C3, C4, C6)
        original_cv = cv2.imread(page.image_path)
        original_gray = cv2.cvtColor(original_cv, cv2.COLOR_BGR2GRAY)
        
        height, width = original_gray.shape
        page_area = height * width
        
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            area = w * h
            
            # Ignore tiny specs of natural noise
            if area < 400: 
                continue
                
            # Crop the bounding box from the ORIGINAL image to see what's actually there
            roi_gray = original_gray[y:y+h, x:x+w]
            
            # Calculate pixel intensity inside the box. (White background = ~255, Dark ink = ~0)
            avg_intensity = np.mean(roi_gray)
            
            # --- Logic Rule: C6 (Watermark Removal) ---
            # If the anomaly is massive (>10% of the page) and mostly empty background
            if area > (page_area * 0.10) and avg_intensity > 230:
                regions.append(DetectedRegion(x=x, y=y, w=w, h=h, category_id="C6"))
                continue
                
            # --- Logic Rule: C4 (Erased Text) ---
            # If the ELA glows, but the original image is just blank white paper (high intensity)
            if avg_intensity > 240:
                regions.append(DetectedRegion(
                    x=x, y=y, w=w, h=h, 
                    category_id="C4", 
                    type="erased"
                ))
            
            # --- Logic Rule: C3 (Added Content) ---
            # If the ELA glows, and the original image contains dark ink/pixels
            elif avg_intensity <= 240:
                # We default the type to 'text', though advanced logic could check for stamps
                regions.append(DetectedRegion(
                    x=x, y=y, w=w, h=h, 
                    category_id="C3", 
                    type="text" 
                ))

    except Exception as e:
        print(f"Tier 3 ELA Error on {page.page_file_name}: {e}")
        
    return regions

In [ ]:
# =========================
# PIPELINE STAGES
# =========================
import cv2
import numpy as np
# async def run_nha_vision_prompt(
#     image_path: str,
#     prompt_text: str,
#     model_name: str = "bedrock/converse/amazon.nova-lite-v1:0",
# ) -> Dict[str, Any]:
#     """
#     Call the NHA client on a page image using a task-specific prompt.

#     Intended responsibilities:
#     - Open the image file from disk and convert it to a base64 data URL.
#     - Send the image and prompt to the mandatory NHA Bedrock model.
#     - Return the raw or lightly normalized response for downstream parsing.
#     - Serve as the common model-calling helper reused across all PS3 subtasks.
#     - Handle API failures, response parsing errors, and retry logic if needed.
#     """
#     pass


def assess_page_quality(page: DocumentPage) -> Dict[str, Any]:
    """
    Assess whether the page image is suitable for tampering analysis using 
    CPU-bound classical computer vision (Laplacian Variance).

    Intended responsibilities:
    - Estimate blur and heavy compression without using ML models.
    - Flag low-quality pages that may reduce confidence.
    - Return a structured quality summary.
    """
    if not page.image_path:
        return {"status": "error", "reason": "No image path provided."}
    
    # Read the image in grayscale (faster and requires less memory)
    img = cv2.imread(page.image_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        return {"status": "error", "reason": "Could not read image file."}
        
    # Calculate the variance of the Laplacian to measure edge sharpness
    laplacian_var = cv2.Laplacian(img, cv2.CV_64F).var()
    
    # Standard threshold: < 100 usually indicates a blurry image
    is_blurry = bool(laplacian_var < 100.0)
    
    # Optional: Check for extreme skew/brightness issues here if needed in the future,
    # but Laplacian variance gives us the best bang-for-buck on a 4-core CPU.
    
    return {
        "laplacian_variance": round(laplacian_var, 2),
        "is_blurry": is_blurry,
        "status": "ok",
        "notes": "Image is blurry" if is_blurry else "Image quality acceptable"
    }


async def detect_tampering_categories(page: DocumentPage) -> List[str]:
    """
    Tier 4 Global Check: Predicts page-level forgery categories (C8).
    
    Intended responsibilities:
    - Runs a Fast Fourier Transform (FFT) to extract the frequency spectrum.
    - Passes the spectrum to an INT8 Quantized ONNX classifier.
    - Returns ["C8"] if fully AI-generated, otherwise returns empty list [].
    """
    categories = []
    if not page.image_path or ort is None:
        return categories
        
    try:
        # 1. Read image in grayscale to save memory
        img = cv2.imread(page.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return categories
            
        # 2. Fast Fourier Transform (FFT) to expose AI frequency artifacts
        f_transform = np.fft.fft2(img)
        f_shift = np.fft.fftshift(f_transform)
        
        # Calculate magnitude spectrum (convert complex numbers to real floats)
        magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1e-8)
        
        # Normalize the spectrum to 0-255 for the model
        cv2.normalize(magnitude_spectrum, magnitude_spectrum, 0, 255, cv2.NORM_MINMAX)
        spectrum_img = np.uint8(magnitude_spectrum)
        
        # Resize to match standard lightweight model input (e.g., 224x224)
        input_tensor = cv2.resize(spectrum_img, (224, 224))
        
        # Prepare for ONNX (Expand dimensions to create batch: Shape [1, 1, 224, 224])
        input_tensor = input_tensor.astype(np.float32) / 255.0
        input_tensor = np.expand_dims(input_tensor, axis=0)
        input_tensor = np.expand_dims(input_tensor, axis=0)
        
        # 3. Quantized ONNX Inference (Extremely fast on CPU)
        # Note: You will need to download an open-source deepfake detection ONNX model 
        # and place it in a 'models' folder in your directory.
        model_path = "./models/efficientnet_lite_int8.onnx"
        
        if Path(model_path).exists():
            session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
            input_name = session.get_inputs()[0].name
            
            # Run the model
            outputs = session.run(None, {input_name: input_tensor})
            ai_probability = float(outputs[0][0])
            
            # Strict threshold to prevent false positives on clean documents
            if ai_probability > 0.85:
                categories.append("C8")
                
    except Exception as e:
        print(f"Tier 4 Frequency Analysis Error on {page.page_file_name}: {e}")

    return categories


async def localize_tampered_regions(
    page: DocumentPage,
    predicted_categories: List[str],
) -> List[DetectedRegion]:
    """
    Detect and localize suspicious regions for categories that require annotations.

    Intended responsibilities:
    - Produce bounding boxes for tampered areas on the page.
    - Associate each bounding box with the corresponding category ID.
    - Ensure boxes tightly cover the manipulated text, stamp, signature, image region,
      spacing anomaly, watermark region, or edited field.
    - Skip localization for category-only classes where YAML is not required.
    """
    pass


def postprocess_regions(
    regions: List[DetectedRegion],
    predicted_categories: List[str],
    page: DocumentPage,
) -> List[DetectedRegion]:
    """
    Clean and normalize the raw localized regions before export.

    Intended responsibilities:
    - Remove invalid or duplicate boxes.
    - Clip coordinates to the image boundaries.
    - Merge overlapping boxes when appropriate.
    - Preserve category-specific metadata such as type, stretch_factor,
      header_source, or body_source when required by sample syntax.
    """
    pass


def enrich_region_metadata(
    regions: List[DetectedRegion],
    predicted_categories: List[str],
    page: DocumentPage,
) -> List[DetectedRegion]:
    """
    Add category-specific annotation metadata required by the organizer examples.

    Intended responsibilities:
    - For C3, optionally tag regions with type values such as text, stamp, or signature.
    - For C4, optionally tag erased regions with a type indicating removed content.
    - For C5, optionally include body_source or header_source values when relevant.
    - For C7, optionally include stretch_factor and a type such as irregular_spacing.
    - For C9, optionally indicate edited field type such as name, date, or amount.
    """
    pass


def validate_prediction(
    page: DocumentPage,
    predicted_categories: List[str],
    regions: List[DetectedRegion],
) -> Dict[str, Any]:
    """
    Validate that the page prediction satisfies submission requirements.

    Intended responsibilities:
    - Ensure that only valid category IDs are used.
    - Enforce that localization exists when the selected category requires YAML.
    - Ensure that category-only classes do not force unnecessary annotations.
    - Confirm that multi-label ordering puts the dominant manipulation first.
    """
    pass


def build_page_analysis_result(
    page: DocumentPage,
    predicted_categories: List[str],
    regions: List[DetectedRegion],
    quality_summary: Optional[Dict[str, Any]] = None,
    validation: Optional[Dict[str, Any]] = None,
) -> PageAnalysisResult:
    """
    Convert page-level outputs into a normalized analysis result object.

    Intended responsibilities:
    - Create a structured result for a single page.
    - Preserve source link, page file name, original file name, page number,
      predicted categories, localized regions, and debug notes.
    - Keep the structure easy to serialize into submission JSON and YAML files.
    """
    pass

## Output Builders

This section should follow the guideline requirements:
- JSON rows contain `link`, `file_name`, and `Category_ID`.
- YAML files are emitted per page using the page-style file name.
- Categories are joined with `||` in the JSON output when multiple apply.
- C8 and C10 do not require YAML. 

In [ ]:
# =========================
# 7. OUTPUT / EXPORT HELPERS 
# =========================

def normalize_category_list(predicted_categories: List[str]) -> List[str]:
    valid = [c for c in predicted_categories if c in CATEGORY_IDS]
    seen = set()
    ordered = []
    for c in valid:
        if c not in seen:
            ordered.append(c)
            seen.add(c)
    return ordered


def build_json_row(result: PageAnalysisResult) -> Dict[str, Any]:
    categories = normalize_category_list(result.predicted_categories)
    category_string = "||".join(categories) if categories else "C10"

    return {
        "link": result.source_link,
        "file_name": result.file_name,
        "Category_ID": category_string,
    }


def detected_region_to_yaml_item(region: DetectedRegion) -> Dict[str, Any]:
    item: Dict[str, Any] = {
        "h": int(region.h),
        "w": int(region.w),
        "x": int(region.x),
        "y": int(region.y),
        "category_id": region.category_id,
    }

    if region.type is not None:
        item["type"] = region.type
    if region.stretch_factor is not None:
        item["stretch_factor"] = region.stretch_factor
    if region.header_source is not None:
        item["header_source"] = region.header_source
    if region.body_source is not None:
        item["body_source"] = region.body_source

    return item


def yaml_required_for_result(result: PageAnalysisResult) -> bool:
    categories = set(normalize_category_list(result.predicted_categories))
    if not categories:
        return False
    return not categories.issubset(CATEGORY_ONLY_CLASSES)


def yaml_name_for_page_file(page_file_name: str) -> str:
    return f"{Path(page_file_name).stem}.yaml"


def write_yaml_for_result(result: PageAnalysisResult, annotations_dir: Path) -> Optional[Path]:
    if not yaml_required_for_result(result):
        return None

    if yaml is None:
        raise ImportError("PyYAML is required to write annotation YAML files.")

    yaml_path = annotations_dir / yaml_name_for_page_file(result.file_name)
    yaml_items = [detected_region_to_yaml_item(region) for region in result.detected_regions]

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_items, f, sort_keys=False, allow_unicode=True)

    return yaml_path


def write_submission_json(results: List[PageAnalysisResult], output_dir: Path) -> Path:
    json_rows = [build_json_row(r) for r in results]
    json_path = output_dir / "submission.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_rows, f, indent=2, ensure_ascii=False)

    return json_path


def write_optional_excel_summary(results: List[PageAnalysisResult], output_dir: Path) -> Optional[Path]:
    if pd is None:
        return None

    rows = []
    for r in results:
        rows.append({
            "link": r.source_link,
            "file_name": r.file_name,
            "Category_ID": "||".join(normalize_category_list(r.predicted_categories)) if r.predicted_categories else "C10",
            "region_count": len(r.detected_regions),
            "page_number": r.page_number,
            "original_file_name": r.original_file_name,
        })

    df = pd.DataFrame(rows)
    excel_path = output_dir / "submission_preview.xlsx"
    df.to_excel(excel_path, index=False)
    return excel_path


def export_all_outputs(results: List[PageAnalysisResult], output_dir: Path, annotations_dir: Path) -> Dict[str, Any]:
    annotation_paths = []
    for result in results:
        yaml_path = write_yaml_for_result(result, annotations_dir)
        if yaml_path is not None:
            annotation_paths.append(str(yaml_path))

    json_path = write_submission_json(results, output_dir)
    excel_path = write_optional_excel_summary(results, output_dir)

    return {
        "json_path": str(json_path),
        "yaml_paths": annotation_paths,
        "excel_preview_path": str(excel_path) if excel_path else None,
        "total_results": len(results),
    }

## Main Runner

The runner is fully implemented. It:
- onboards files,
- expands PDFs to page images,
- runs page-wise pipeline stages,
- creates final page results,
- exports JSON and YAML outputs.

The actual detection stages remain empty and will fall back to scaffold behavior until implemented.

In [ ]:
# =========================
# 9. MAIN RUNNER (FILLED)
# =========================

async def run_ps3_pipeline(
    input_dir: Path = INPUT_DIR,
    output_dir: Path = OUTPUT_DIR,
    annotations_dir: Path = ANNOTATIONS_DIR,
    debug_dir: Path = DEBUG_DIR,
) -> Dict[str, Any]:
    render_dir = debug_dir / "rendered_pages"
    pages = build_document_pages(input_dir, render_dir)

    results: List[PageAnalysisResult] = []

    for page in pages:
        try:
            quality_summary = assess_page_quality(page)
            if quality_summary is None:
                quality_summary = fallback_quality_summary(page)
        except Exception:
            quality_summary = fallback_quality_summary(page)

        try:
            predicted_categories = await detect_tampering_categories(page)
            if predicted_categories is None:
                predicted_categories = fallback_predicted_categories(page)
        except Exception:
            predicted_categories = fallback_predicted_categories(page)

        predicted_categories = normalize_category_list(predicted_categories) or ["C10"]

        try:
            regions = await localize_tampered_regions(page, predicted_categories)
            if regions is None:
                regions = fallback_regions(page, predicted_categories)
        except Exception:
            regions = fallback_regions(page, predicted_categories)

        try:
            regions = postprocess_regions(regions, predicted_categories, page)
            if regions is None:
                regions = fallback_regions(page, predicted_categories)
        except Exception:
            regions = fallback_regions(page, predicted_categories)

        try:
            regions = enrich_region_metadata(regions, predicted_categories, page)
            if regions is None:
                regions = fallback_regions(page, predicted_categories)
        except Exception:
            regions = fallback_regions(page, predicted_categories)

        try:
            validation = validate_prediction(page, predicted_categories, regions)
            if validation is None:
                validation = fallback_validation(page, predicted_categories, regions)
        except Exception:
            validation = fallback_validation(page, predicted_categories, regions)

        try:
            result = build_page_analysis_result(
                page=page,
                predicted_categories=predicted_categories,
                regions=regions,
                quality_summary=quality_summary,
                validation=validation,
            )
            if result is None:
                result = fallback_build_result(page, predicted_categories, regions, quality_summary, validation)
        except Exception:
            result = fallback_build_result(page, predicted_categories, regions, quality_summary, validation)

        results.append(result)

    export_info = export_all_outputs(results, output_dir, annotations_dir)

    return {
        "pages": pages,
        "results": results,
        "export_info": export_info,
    }

## Display Helpers

In [ ]:
# =========================
# 10. DISPLAY FINAL OUTPUTS (FILLED)
# =========================

def display_final_outputs(run_output: Dict[str, Any]) -> None:
    results: List[PageAnalysisResult] = run_output.get("results", [])
    export_info: Dict[str, Any] = run_output.get("export_info", {})

    print("=" * 80)
    print("PS3 PIPELINE RUN SUMMARY")
    print("=" * 80)
    print(f"Total page-level entries processed: {len(results)}")
    print(f"Submission JSON: {export_info.get('json_path')}")
    print(f"YAML files written: {len(export_info.get('yaml_paths', []))}")
    if export_info.get("excel_preview_path"):
        print(f"Excel preview: {export_info.get('excel_preview_path')}")
    print()

    preview_rows = [build_json_row(r) for r in results[:10]]
    if pd is not None and preview_rows:
        display(pd.DataFrame(preview_rows))
    else:
        print("Preview JSON rows:")
        print(json.dumps(preview_rows, indent=2))

    if export_info.get("yaml_paths"):
        print()
        print("Sample YAML files:")
        for yp in export_info["yaml_paths"][:10]:
            print("-", yp)

## Execution Cell



In [ ]:
# =========================
# 11. EXECUTE
# =========================

run_output = await run_ps3_pipeline()
display_final_outputs(run_output)